#  UTD19 Torino traffic loop detectors — Cleaning + Qdrant + Neo4j

**City Brain · traffic.** Road sensors that report flow, occupancy and speed every 5 minutes (399 detectors, 21 normal days, Torino 2016).
This one notebook does everything for this dataset:

1. **Load & clean** the Torino extract
2. **Visualise** rush hour and where jams happen
3. **Turn every detector-hour into a small numeric vector**: *how slow, how busy, how full is the road compared with its own normal?*
4. **Find jam episodes and incidents** (a jam seen by several neighbouring detectors at once is a real jam; a single detector is probably a glitch)
5. **Qdrant** (cloud): store the vectors, so a new hour of readings can be compared with past ones
6. **Neo4j** (cloud): store the road network and the jam structure, to explain *why*
7. **Combine**: a new reading → Qdrant finds similar past hours → Neo4j says whether those were real, confirmed jams

### What goes where

| Store | Holds | Answers |
|---|---|---|
| **Qdrant** | one **6-number vector** per detector-hour (speed vs normal: mean/min, occupancy: mean/max, flow vs capacity, flow vs normal) + a small payload (`state`, a few readings) | *"Which past hours looked like this one?"* |
| **Neo4j** | `Detector → Link → Road`, `NEXT_TO` neighbours, `JamEpisode`, `Incident`, `Reading` | *"Was it a real jam (several neighbours at once) or one sensor? Where? How long did similar ones last?"* |

The vectors are **relative to each detector's own normal** (not raw km/h), so the same encoding can be applied to another city's detectors once you give it their baselines.

### Graph model

| Node | Meaning |
|---|---|
| `Detector` | a loop detector: road, class, lanes, position |
| `Link`, `Road` | the road segment and street it sits on |
| `JamEpisode` | one detector in a jam for >= 30 min (speed < 50 % of its normal for that hour **and** occupancy >= 25 %) |
| `Incident` | jam episodes of **neighbouring** detectors that overlap in time, merged into one. `confirmed = true` when >= 2 detectors saw it |
| `IncidentType` | `traffic_jam` (shared with the other City Brain datasets) |
| `Reading` | one detector-hour (same id as the Qdrant point) |

Relationships: `Detector-[:NEXT_TO {dist_m}]-Detector`, `JamEpisode-[:AT]->Detector`, `JamEpisode-[:PART_OF]->Incident`, `Incident-[:IS_A]->IncidentType`, `Reading-[:AT]->Detector`, `Reading-[:IN_EPISODE]->JamEpisode`.

> Some detectors are kept **out** of Qdrant and Neo4j readings (`HOLDOUT_FRAC`). They play the role of *"new sensors the system has never seen"*.
> There are **no labelled incidents** in this dataset. Jams here are defined by a rule, so numbers like "accuracy of the state" only measure consistency; the useful test is whether the neighbours of a new jam tell you if it was **confirmed by several detectors** (section 4).

## 0 · Setup

In [ ]:
import os, io, json, re, sys, subprocess, zipfile, warnings
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import display

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print("Running in Colab:", IN_COLAB)

In [ ]:
import uuid

# ---- switches ----------------------------------------------------------------
RUN_QDRANT      = False   # section 5: upload the vectors to your Qdrant Cloud cluster
RUN_NEO4J       = False   # section 6: load the graph into your Neo4j (Aura) instance
RUN_COMBINE     = False   # section 7: query Qdrant, explain with Neo4j (needs both uploads done)
RECREATE        = False   # True = delete the Qdrant collection first (destroys points already in it)
CLEAR_GRAPH     = False   # True = delete the nodes this notebook created earlier in Neo4j (tag ds = DS)
INCLUDE_CONTEXT = False   # True = also store detector id / day / hour / road in the Qdrant payload

# ---- data location -------------------------------------------------------------
DATA_DIR        = ""      # e.g. "/content/drive/MyDrive/utd19" if you mounted Google Drive; "" = current folder / upload dialog
MAX_DAYS        = None    # e.g. 7 to test quickly on the first 7 days
MAX_DETECTORS   = None    # e.g. 100 to test quickly

# ---- parameters ---------------------------------------------------------------
HOLDOUT_FRAC    = 0.08    # share of detectors kept OUT of the databases (the "unseen sensors")
JAM_MIN_RUN     = 6       # >= 6 consecutive 5-min intervals (30 min) = a jam episode
NEIGHBOUR_M     = 400     # detectors closer than this are neighbours...
NEIGHBOUR_K     = 4       # ...(at most this many per detector)
PER_STATE_CAP   = {"jam": None, "slow": 5000, "free_flow": 6000, "night_empty": 3000}   # points per state sent to Qdrant/Neo4j
K               = 15      # neighbours per query
DS              = "utd19-torino"            # tag on every Neo4j node we create
COLLECTION      = "utd19_torino_windows"    # Qdrant collection
SEED            = 0

In [ ]:
INK, INK2, SURF, GRIDC = "#0b0b0b", "#52514e", "#fcfcfb", "#e4e3df"
SEQ = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"])
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": SURF, "axes.facecolor": SURF, "savefig.facecolor": SURF,
    "axes.edgecolor": GRIDC, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.grid": True, "grid.color": GRIDC, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "font.size": 10,
})

In [ ]:
STATE_ORDER  = ["free_flow", "slow", "jam", "night_empty"]
STATE_COLORS = {"free_flow": "#2a78d6", "slow": "#1baf7a", "jam": "#eb6834", "night_empty": "#9a9992"}
WEEK_COLORS  = {"weekday": "#2a78d6", "weekend": "#eb6834"}

## 1 · Load the Torino extract

You need two files from your UTD19 download (the 7 GB `utd19_u.csv` is **not** needed here):

* `torino.csv`: the Torino rows (`awk -F, '$7=="torino"' utd19_u.csv > torino.csv`, keep the header line)
* `torino_dets.csv`: the detector table (`detectors_public.csv` filtered to Torino)

`torino.csv` is ~2 million rows: uploading it in Colab works but is slow, so **mounting Google Drive** and setting `DATA_DIR` is faster.
UTD19 is free for **academic / non-commercial use only**: the data stays in your own Colab session, your own Qdrant and your own Neo4j.

In [ ]:
def find_file(name):
    for d in [Path(DATA_DIR) if DATA_DIR else None, Path(".")]:
        if d is not None and (d / name).exists():
            return d / name
    return None

NEEDED = ["torino.csv", "torino_dets.csv"]
missing = [n for n in NEEDED if find_file(n) is None]
if missing:
    if IN_COLAB:
        from google.colab import files
        print("Select", ", ".join(missing), "...")
        files.upload()
        missing = [n for n in NEEDED if find_file(n) is None]
    if missing:
        raise FileNotFoundError(f"Missing {missing}. Put them next to the notebook or set DATA_DIR.")

raw = pd.read_csv(find_file("torino.csv"), usecols=lambda c: c in {"day", "interval", "detid", "flow", "occ", "error", "speed"})
dets = pd.read_csv(find_file("torino_dets.csv"))
dets = dets.rename(columns={"long": "lon"})
print(f"{len(raw):,} rows | {raw.detid.nunique()} detectors | {raw.day.nunique()} days ({raw.day.min()} -> {raw.day.max()})")
print("detector table columns:", list(dets.columns))
raw.head()

## 2 · Cleaning

| Issue | Treatment |
|---|---|
| a few occupancy values above 100 % (impossible) | dropped |
| rows flagged by the `error` column | dropped |
| negative flow / occupancy / speed, speed above 150 km/h | dropped |
| **speed missing** (only 9 UTD19 cities have it) | rows without speed are dropped: the jam rule needs speed |
| duplicates / unsorted | one row per detector + day + 5-min interval, sorted |
| detectors without road name or position | kept, road = `unknown`, no neighbours |
| `day` is a string | `day_idx` (days since the first day) and `abs_s` (seconds since the first midnight) |

In [ ]:
d = raw.copy()
n0 = len(d)
steps = {}

def drop(mask, label):
    global d
    steps[label] = int(mask.sum()); d = d[~mask]

if MAX_DAYS:
    keep_days = sorted(d.day.unique())[:MAX_DAYS]; d = d[d.day.isin(keep_days)]
if MAX_DETECTORS:
    keep_dets = sorted(d.detid.unique())[:MAX_DETECTORS]; d = d[d.detid.isin(keep_dets)]
n0 = len(d)

drop(d.occ.gt(100) | d.occ.lt(0), "occupancy outside 0-100 %")
if "error" in d:
    drop(d.error.notna() & (pd.to_numeric(d.error, errors="coerce").fillna(0) > 0), "flagged by the error column")
drop(d.flow.lt(0), "negative flow")
drop(d.speed.isna(), "speed missing")
drop(d.speed.lt(0) | d.speed.gt(150), "speed outside 0-150 km/h")
drop(d.duplicated(["detid", "day", "interval"]), "duplicate detector/day/interval")

d = d.sort_values(["detid", "day", "interval"]).reset_index(drop=True)
day0 = pd.to_datetime(d.day).min()
d["day_idx"] = (pd.to_datetime(d.day) - day0).dt.days.astype("int32")
d["abs_s"]   = d.day_idx.astype("int64") * 86400 + d.interval
d["hour"]    = (d.interval // 3600).astype("int8")
d["weekend"] = (pd.to_datetime(d.day).dt.dayofweek >= 5)
d = d.astype({"detid": "int32", "interval": "int32", "flow": "float32", "occ": "float32", "speed": "float32"}).drop(columns=["error"], errors="ignore")

print(f"{n0:,} rows in -> {len(d):,} rows kept ({len(d) / n0 * 100:.1f}%)")
display(pd.Series(steps, name="rows dropped").to_frame())
print(f"{d.detid.nunique()} detectors | {d.day.nunique()} days | {int(d.weekend.groupby(d.day).first().sum())} weekend days")

### Rush hour: what normal traffic looks like

In [ ]:
prof = d.assign(kind=np.where(d.weekend, "weekend", "weekday")).groupby(["kind", "hour"])[["flow", "speed", "occ"]].median().unstack(0)
fig, ax = plt.subplots(1, 3, figsize=(14, 3.4))
for i, (c, t) in enumerate([("flow", "Flow (veh/h)"), ("speed", "Speed (km/h)"), ("occ", "Occupancy (%)")]):
    for kind, col in WEEK_COLORS.items():
        ax[i].plot(prof[c][kind].index, prof[c][kind].values, color=col, lw=2, marker="o", ms=3, label=kind)
    ax[i].set_title(t); ax[i].set_xlabel("hour of day")
ax[0].legend(fontsize=8.5)
wd = prof["flow"]["weekday"]
fig.suptitle(f"Weekday flow peaks at {int(wd[wd.index < 12].idxmax())}h and {int(wd[wd.index >= 12].idxmax())}h, weekends have no morning peak: "
             "every detector needs its own per-hour baseline", x=0.01, ha="left", fontsize=11, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.93]); plt.show()

## 3 · Every detector is compared with its *own* normal

For each detector, each hour of day and weekday/weekend, we compute the **normal speed and flow** (median over all days).
Then every 5-minute reading becomes:

* `speed_ratio` = speed / normal speed (1.0 = normal, 0.5 = half as fast as usual)
* `flow_norm` = flow / the detector's 95th-percentile flow (how full the road is; ~0 at night)
* `flow_ratio` = flow / normal flow for that hour

A reading is a **jam interval** when `speed_ratio < 0.5` **and** `occ >= 25 %` **and** the road is not empty (`flow_norm >= 0.05`, otherwise speed means nothing).
Runs of at least `JAM_MIN_RUN` = 6 consecutive jam intervals (30 min) are **jam episodes**. This is the rule that produced the fewest false alarms in the project's study of these 21 normal days.

In [ ]:
G = ["detid", "weekend", "hour"]
base = d.groupby(G, observed=True).agg(base_speed=("speed", "median"), base_flow=("flow", "median")).reset_index()
p95  = d.groupby("detid").flow.quantile(0.95).rename("p95_flow").reset_index()
d = d.join(base.set_index(G), on=G).join(p95.set_index("detid"), on="detid")

d["speed_ratio"] = np.where(d.base_speed > 1, d.speed / d.base_speed, 1.0).astype("float32")
d["flow_norm"]   = (d.flow / d.p95_flow.clip(lower=1)).astype("float32")
d["flow_ratio"]  = (d.flow / d.base_flow.clip(lower=1)).clip(upper=5).astype("float32")

cond = (d.speed_ratio < 0.5) & (d.occ >= 25) & (d.flow_norm >= 0.05)
key  = d.detid.astype("int64") * 10_000 + d.day_idx
new_run = (cond != cond.shift()) | (key != key.shift()) | (d.interval.diff() != 300)
run_id = new_run.cumsum()
run_len = d.groupby(run_id)["interval"].transform("size")
d["in_jam"] = cond & (run_len >= JAM_MIN_RUN)
d["run"]    = run_id
print(f"{int(cond.sum()):,} jam intervals before the 30-min rule -> {int(d.in_jam.sum()):,} after ({d.in_jam.mean() * 100:.2f}% of all intervals)")

# ---------------- jam episodes ----------------
ep = (d[d.in_jam].groupby("run").agg(detid=("detid", "first"), day=("day", "first"), day_idx=("day_idx", "first"),
        start_i=("interval", "min"), end_i=("interval", "max"), n=("interval", "size"),
        min_speed_ratio=("speed_ratio", "min"), max_occ=("occ", "max")).reset_index(drop=True))
ep["start_s"] = ep.day_idx.astype("int64") * 86400 + ep.start_i
ep["end_s"]   = ep.day_idx.astype("int64") * 86400 + ep.end_i + 300
ep["minutes"] = ep.n * 5
ep["id"]      = ["J" + str(i + 1) for i in range(len(ep))]
ep["start"]   = [f"{a} {s // 3600:02d}:{s % 3600 // 60:02d}" for a, s in zip(ep.day, ep.start_i)]
ep["end"]     = [f"{a} {(s + 300) // 3600 % 24:02d}:{(s + 300) % 3600 // 60:02d}" for a, s in zip(ep.day, ep.end_i)]
print(f"{len(ep):,} jam episodes on {ep.detid.nunique()} detectors "
      f"({len(ep) / (d.detid.nunique() * d.day.nunique()):.3f} per detector-day)  |  median length {ep.minutes.median():.0f} min")
ep.head()

### Neighbouring detectors and incidents

Detectors closer than `NEIGHBOUR_M` are linked (`NEXT_TO`). Jam episodes on neighbouring detectors that **overlap in time** are merged into one **incident**.
`confirmed = True` means at least 2 detectors saw it, the project's rule for a real jam. A single-detector episode is probably a sensor glitch or a very local slowdown.

In [ ]:
dets = dets.drop_duplicates("detid").copy()
for c_, default in [("road", "unknown"), ("fclass", "unknown")]:
    if c_ not in dets: dets[c_] = default
    dets[c_] = dets[c_].fillna(default).astype(str)
for c_ in ("lanes", "limit", "linkid"):
    if c_ not in dets: dets[c_] = np.nan
used = dets[dets.detid.isin(d.detid.unique())].reset_index(drop=True)
has_pos = used.lat.notna() & used.lon.notna()

def haversine_m(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 6371000 * 2 * np.arcsin(np.sqrt(a))

pairs = []
P = used[has_pos].reset_index(drop=True)
if len(P) > 1:
    D = haversine_m(P.lat.to_numpy()[:, None], P.lon.to_numpy()[:, None], P.lat.to_numpy()[None, :], P.lon.to_numpy()[None, :])
    np.fill_diagonal(D, np.inf)
    for i in range(len(P)):
        for j in np.argsort(D[i])[:NEIGHBOUR_K]:
            if D[i, j] <= NEIGHBOUR_M:
                pairs.append((int(P.detid[i]), int(P.detid[j]), float(D[i, j])))
neigh = pd.DataFrame(pairs, columns=["a", "b", "dist_m"])
print(f"{len(used)} detectors with data ({int(has_pos.sum())} with a position) | {len(neigh)} NEXT_TO links "
      f"| median distance {neigh.dist_m.median():.0f} m" if len(neigh) else "no neighbour links (no positions)")

# union-find over episodes: overlapping in time on neighbouring detectors
parent = list(range(len(ep)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
def union(i, j):
    ri, rj = find(i), find(j)
    if ri != rj: parent[rj] = ri

by_det = {k: g for k, g in ep.groupby("detid")}
for a, b in {tuple(sorted(p)) for p in zip(neigh.a, neigh.b)} if len(neigh) else []:
    A, B = by_det.get(a), by_det.get(b)
    if A is None or B is None: continue
    ov = (A.start_s.to_numpy()[:, None] < B.end_s.to_numpy()[None, :]) & (B.start_s.to_numpy()[None, :] < A.end_s.to_numpy()[:, None])
    for i, j in zip(*np.nonzero(ov)):
        union(A.index[i], B.index[j])

ep["comp"] = [find(i) for i in range(len(ep))]
road_of = dict(zip(used.detid, used.road))
inc = (ep.groupby("comp").agg(day=("day", "first"), start_s=("start_s", "min"), end_s=("end_s", "max"), n_episodes=("id", "size"),
                              n_detectors=("detid", "nunique"), min_speed_ratio=("min_speed_ratio", "min"),
                              roads=("detid", lambda s: ", ".join(sorted({road_of.get(x, "unknown") for x in s})[:3]))).reset_index())
inc["id"]        = ["I" + str(i + 1) for i in range(len(inc))]
inc["minutes"]   = ((inc.end_s - inc.start_s) / 60).round().astype(int)
inc["confirmed"] = inc.n_detectors >= 2
ep["incident"]   = ep.comp.map(dict(zip(inc.comp, inc.id)))
ep = ep.drop(columns="comp")
print(f"{len(inc):,} incidents | confirmed by >= 2 neighbouring detectors: {int(inc.confirmed.sum())} ({inc.confirmed.mean() * 100:.0f}%) | single detector: {int((~inc.confirmed).sum())}")
inc.drop(columns=["comp"]).sort_values("n_detectors", ascending=False).head(5)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.4), gridspec_kw={"width_ratios": [1.3, 1]})
h = ep.start_i // 3600
ax[0].bar(*np.unique(h, return_counts=True), color="#2a78d6", edgecolor=SURF)
ax[0].set_xlabel("hour a jam episode starts"); ax[0].set_ylabel("episodes")
ax[0].set_title("Jam episodes start in the morning and evening rush")
cnt = ep.groupby("detid").size()
pos = used.set_index("detid")[["lat", "lon"]].dropna()
ax[1].scatter(pos.lon, pos.lat, s=14, color="#cde2fb", edgecolor="#86b6ef", lw=0.5, label="detector")
c_ = cnt.reindex(pos.index).fillna(0)
m = c_ > 0
ax[1].scatter(pos.lon[m], pos.lat[m], s=20 + c_[m] * 12, color="#2a78d6", alpha=0.7, edgecolor=SURF, lw=0.8, label="jam episodes (size = count)")
ax[1].set_aspect(1 / np.cos(np.deg2rad(pos.lat.mean()))); ax[1].grid(False); ax[1].set_xticks([]); ax[1].set_yticks([])
for s_ in ax[1].spines.values(): s_.set_visible(False)
ax[1].set_title("Where the jams are"); ax[1].legend(fontsize=8, loc="lower left")
plt.tight_layout(); plt.show()

### One vector per detector-hour

Each (detector, day, hour) with at least 8 readings becomes a **window** described by 6 numbers:

| # | Feature | Meaning |
|---|---|---|
| 1-2 | `speed_ratio_mean`, `speed_ratio_min` | how slow vs this detector's normal (average / worst 5 min) |
| 3-4 | `occ_mean`, `occ_max` | occupancy (%) |
| 5 | `flow_norm_mean` | how full the road is vs its capacity-like 95th percentile |
| 6 | `flow_ratio_mean` | flow vs normal for that hour |

All are clipped and divided by **fixed constants** (not fitted to this data), so another city's windows can be encoded identically. Distance in Qdrant: **Euclidean**.

Weak label `state` (by rule): `jam` (>= 6 jam intervals in the hour) > `night_empty` (almost no cars) > `slow` (speed < 70 % of normal or occupancy >= 15 %) > `free_flow`.

In [ ]:
agg = (d.groupby(["detid", "day_idx", "hour"], observed=True)
         .agg(day=("day", "first"), weekend=("weekend", "first"), n=("speed", "size"),
              speed_ratio_mean=("speed_ratio", "mean"), speed_ratio_min=("speed_ratio", "min"),
              occ_mean=("occ", "mean"), occ_max=("occ", "max"), flow_norm_mean=("flow_norm", "mean"),
              flow_ratio_mean=("flow_ratio", "mean"), n_jam=("in_jam", "sum")).reset_index())
agg = agg[agg.n >= 8].reset_index(drop=True)
agg["state"] = np.select([agg.n_jam >= 6, agg.flow_norm_mean < 0.05, (agg.speed_ratio_mean < 0.7) | (agg.occ_mean >= 15)],
                         ["jam", "night_empty", "slow"], default="free_flow")

VEC_FEATURES = ["speed_ratio_mean", "speed_ratio_min", "occ_mean", "occ_max", "flow_norm_mean", "flow_ratio_mean"]
def encode(f):
    """raw window features (DataFrame or dict) -> (n, 6) vectors with fixed scaling."""
    f = pd.DataFrame(f if not isinstance(f, dict) else [f])
    return np.nan_to_num(np.c_[
        f.speed_ratio_mean.clip(0, 1.5), f.speed_ratio_min.clip(0, 1.5),
        (f.occ_mean / 50).clip(0, 2), (f.occ_max / 50).clip(0, 2),
        f.flow_norm_mean.clip(0, 1.5), f.flow_ratio_mean.clip(0, 3) / 2]).astype("float32")

# which jam episode / incident does each window belong to (for the "confirmed" truth)
we = agg[["detid", "day_idx", "hour"]].reset_index().merge(ep[["detid", "day_idx", "start_s", "end_s", "id", "incident"]], on=["detid", "day_idx"])
we["w0"] = we.day_idx.astype("int64") * 86400 + we.hour.astype("int64") * 3600
we["ov"] = np.minimum(we.end_s, we.w0 + 3600) - np.maximum(we.start_s, we.w0)
we = we[we.ov > 0].sort_values("ov", ascending=False).drop_duplicates("index").set_index("index")
agg["episode"]  = we.id.reindex(agg.index)
agg["incident"] = we.incident.reindex(agg.index)
agg["confirmed"] = agg.incident.map(dict(zip(inc.id, inc.confirmed)))

rng = np.random.default_rng(SEED)
det_ids = np.array(sorted(agg.detid.unique()))
test_dets = set(rng.choice(det_ids, size=max(1, int(len(det_ids) * HOLDOUT_FRAC)), replace=False).tolist())
agg["holdout"] = agg.detid.isin(test_dets)
print(f"{len(agg):,} windows | held-out detectors: {len(test_dets)} ({int(agg.holdout.sum()):,} windows)")
display(agg.groupby("state").size().reindex(STATE_ORDER).rename("windows").to_frame())

In [ ]:
samp = agg.sample(min(12000, len(agg)), random_state=SEED)
Xs = encode(samp); Xc = Xs - Xs.mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False); Pp = Xc @ Vt[:2].T
fig, ax = plt.subplots(figsize=(7.5, 5))
for st in ["night_empty", "free_flow", "slow", "jam"]:
    m = (samp.state == st).to_numpy()
    ax.scatter(Pp[m, 0], Pp[m, 1], s=6, color=STATE_COLORS[st], alpha=0.5, lw=0, label=f"{st} ({m.sum():,})")
ax.set_xlabel(f"component 1 ({S[0]**2 / (S**2).sum() * 100:.0f}% of variance)"); ax.set_ylabel(f"component 2 ({S[1]**2 / (S**2).sum() * 100:.0f}%)")
ax.set_title("Detector-hours form regions: jam, slow, free flow, empty night road")
ax.legend(markerscale=3, fontsize=9)
plt.tight_layout(); plt.show()

## 4 · Choose the points to store, and check the search offline

* **What is stored**: every `jam` window, plus a random sample of the other states up to `PER_STATE_CAP` (otherwise ~170,000 windows, mostly quiet ones, would fill your free cloud tiers). Held-out detectors are never stored.
* **Offline check (no cloud)**, using exactly what Qdrant will do (15 nearest windows by Euclidean distance):
  1. *consistency*: do neighbours of a held-out window have the same state? (the state is rule-based, so this only shows the vectors keep the rule's information)
  2. **the useful test**: for a held-out **jam** window, the neighbours that were jams - were they confirmed by several detectors? Does that predict whether this jam was confirmed? (`confirmed` comes from *other* detectors, which the vector never sees)

In [ ]:
pools = []
for st, g in agg[~agg.holdout].groupby("state"):
    cap = PER_STATE_CAP.get(st)
    pools.append(g if cap is None or len(g) <= cap else g.sample(cap, random_state=SEED))
train = pd.concat(pools).sort_values(["detid", "day_idx", "hour"]).reset_index(drop=True)
train["point_id"] = [str(uuid.uuid5(uuid.NAMESPACE_URL, f"utd19|{a}|{b}|{c}")) for a, b, c in zip(train.detid, train.day, train.hour)]
assert train.point_id.is_unique
Xtr = encode(train)
ho  = agg[agg.holdout].reset_index(drop=True)
Xho = encode(ho)
print(f"to store: {len(train):,} windows  ->  {train.state.value_counts().reindex(STATE_ORDER).to_dict()}")

def knn_idx(Xq, Xdb, k=K, chunk=1000):
    sqd = (Xdb.astype(float) ** 2).sum(1)
    out = np.empty((len(Xq), k), dtype=int)
    for i in range(0, len(Xq), chunk):
        xq = Xq[i:i + chunk].astype(float)
        dd = np.maximum((xq ** 2).sum(1)[:, None] + sqd[None, :] - 2 * xq @ Xdb.T.astype(float), 0)
        out[i:i + chunk] = np.argpartition(dd, k, axis=1)[:, :k]
    return out

def auc(score, label):
    label = np.asarray(label, bool); n1, n0 = label.sum(), (~label).sum()
    if n1 == 0 or n0 == 0: return np.nan
    r = pd.Series(score).rank().to_numpy()
    return float((r[label].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

# 1) consistency of state
q_ho = ho.sample(min(4000, len(ho)), random_state=SEED) if len(ho) else ho
if len(q_ho):
    nn = knn_idx(encode(q_ho), Xtr)
    ytr = train.state.to_numpy()
    pred = [Counter(ytr[r]).most_common(1)[0][0] for r in nn]
    print(f"\n1) state consistency on {len(q_ho):,} held-out windows: {(np.array(pred) == q_ho.state.to_numpy()).mean() * 100:.1f}%")
    display(pd.crosstab(q_ho.state.reset_index(drop=True), pd.Series(pred, name="neighbours say")))

# 2) do jam neighbours predict "confirmed by neighbours"?
hj = ho[(ho.state == "jam") & ho.confirmed.notna()].reset_index(drop=True)
tj = train[(train.state == "jam") & train.confirmed.notna()].reset_index(drop=True)
if len(hj) >= 5 and len(tj) >= K:
    nn = knn_idx(encode(hj), encode(tj))
    score = np.array([tj.confirmed.astype(float).to_numpy()[r].mean() for r in nn])
    truth = hj.confirmed.astype(bool).to_numpy()
    print(f"\n2) held-out jam windows: {len(hj)} | really confirmed by >= 2 detectors: {truth.mean() * 100:.0f}%")
    print(f"   average neighbour score  when truly confirmed: {score[truth].mean():.2f}   when single-detector: {score[~truth].mean() if (~truth).any() else float('nan'):.2f}")
    print(f"   AUC (0.5 = coin flip, 1.0 = perfect): {auc(score, truth):.2f}")
else:
    print("\n2) not enough held-out jam windows with a known incident to evaluate (try a smaller HOLDOUT_FRAC or more days).")

> **How to read this honestly.** One city, 21 normal days, no labelled incidents: "jam" is a rule, not ground truth, and "confirmed" only means *several neighbouring detectors agreed*. A good AUC in test 2 means the **shape** of a jam hour tells you something about whether it is a real, network-wide jam; it does not prove anything about accidents. For another city, feed in that city's baselines (see the `diagnose` helper) and re-check.

## 5 · Qdrant — store the vectors

* vector = the 6 encoded numbers, distance = **Euclidean**
* point id = a fixed UUID from `detector + day + hour` (same id as the Neo4j `Reading` node)
* payload is small: `state`, `speed_ratio`, `occ`, `flow_norm`, `dataset`. With `INCLUDE_CONTEXT = True` it also stores `detid`, `day`, `hour`, `weekend`, `road`.

Credentials: Colab **Secrets** `QDRANT_URL` and `QDRANT_API_KEY` (with *Notebook access* on). If missing, the cell asks (key typed hidden).

In [ ]:
def secret(name, prompt, hidden=False, default=None):
    """Colab Secrets -> environment variable -> prompt. Nothing is stored in the notebook."""
    import getpass
    val = None
    if IN_COLAB:
        try:
            from google.colab import userdata
            val = userdata.get(name)
        except Exception:
            val = None
    val = val or os.environ.get(name)
    if not val and default is None:
        val = (getpass.getpass if hidden else input)(prompt + ": ")
    return (val or default).strip()

In [ ]:
_qdrant = {}

def get_qdrant():
    """Connect to your Qdrant Cloud cluster once (URL + API key from Colab Secrets)."""
    if "client" not in _qdrant:
        try:
            import qdrant_client  # noqa: F401
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "qdrant-client"], check=True)
        from qdrant_client import QdrantClient
        url = secret("QDRANT_URL", "Qdrant URL (from the Qdrant Cloud dashboard)").rstrip("/")
        key = secret("QDRANT_API_KEY", "Qdrant API key", hidden=True)
        client = QdrantClient(url=url, api_key=key, timeout=60)
        try:
            client.get_collections()
        except Exception as e:
            raise RuntimeError(f"Could not reach Qdrant at {url}. Check the URL (https://..., port 6333 if needed) and the API key. Original error: {e}")
        _qdrant["client"] = client
    return _qdrant["client"]

def upload_points(collection, ids, vectors, payloads, distance="Euclid", keyword_indexes=(), recreate=False, batch=256):
    """Create the collection if needed (never deleted unless recreate=True) and upsert the points in batches."""
    from qdrant_client import models
    client = get_qdrant()
    dist = {"Euclid": models.Distance.EUCLID, "Cosine": models.Distance.COSINE}[distance]
    dim = int(vectors.shape[1])
    existing = [c.name for c in client.get_collections().collections]
    print("Collections on the cluster:", existing)
    if collection in existing and recreate:
        client.delete_collection(collection); existing.remove(collection)
    if collection not in existing:
        client.create_collection(collection, vectors_config=models.VectorParams(size=dim, distance=dist))
        print(f"Created collection '{collection}' ({dim} dims, {distance})")
    else:
        cfg = client.get_collection(collection).config.params.vectors
        if getattr(cfg, "size", dim) != dim:
            raise ValueError(f"Collection '{collection}' has {cfg.size} dims but the vectors have {dim}. Use another name or recreate=True.")
        print(f"Collection '{collection}' already exists - upserting into it")
    for field in keyword_indexes:
        client.create_payload_index(collection, field_name=field, field_schema="keyword")
    for i in range(0, len(ids), batch):
        client.upsert(collection, wait=True, points=[
            models.PointStruct(id=ids[j], vector=vectors[j].tolist(), payload=payloads[j])
            for j in range(i, min(i + batch, len(ids)))])
        if (i // batch) % 10 == 0:
            print(f"  uploaded {min(i + batch, len(ids)):,} / {len(ids):,}")
    print("points in collection:", client.count(collection, exact=True).count)

def qdrant_search(collection, vector, k=15, query_filter=None):
    """k nearest stored points: list of (id, distance, payload). With Euclid, smaller distance = more similar."""
    client = get_qdrant()
    res = client.query_points(collection, query=[float(v) for v in vector], limit=k, query_filter=query_filter, with_payload=True).points
    return [(str(h.id), float(h.score), h.payload) for h in res]

def qdrant_search_batch(collection, vectors, k=15, chunk=64):
    """Many queries at once -> list of lists of (id, distance, payload)."""
    from qdrant_client import models
    client = get_qdrant()
    out = []
    for i in range(0, len(vectors), chunk):
        reqs = [models.QueryRequest(query=[float(v) for v in vec], limit=k, with_payload=True) for vec in vectors[i:i + chunk]]
        for r in client.query_batch_points(collection, requests=reqs):
            out.append([(str(h.id), float(h.score), h.payload) for h in r.points])
    return out

In [ ]:
road_of = dict(zip(used.detid, used.road))

def payload_of(r):
    p = {"state": r.state, "speed_ratio": round(float(r.speed_ratio_mean), 3), "occ": round(float(r.occ_mean), 2),
         "flow_norm": round(float(r.flow_norm_mean), 3), "dataset": "utd19-torino"}
    if INCLUDE_CONTEXT:
        p.update({"detid": int(r.detid), "day": r.day, "hour": int(r.hour), "weekend": bool(r.weekend), "road": road_of.get(r.detid, "unknown")})
    return p

payloads = [payload_of(r) for r in train.itertuples()]
print(f"{len(train):,} points ready (vector size {Xtr.shape[1]}). Example payload:\n", json.dumps(payloads[len(payloads) // 2], indent=2))

if RUN_QDRANT:
    upload_points(COLLECTION, train.point_id.tolist(), Xtr, payloads, distance="Euclid", keyword_indexes=["state"], recreate=RECREATE)
else:
    print("\nRUN_QDRANT is False - skipping the upload.")

## 6 · Neo4j — store the road network and the jam structure

Add in Colab **Secrets**: `NEO4J_URI` (`neo4j+s://xxxx.databases.neo4j.io`), `NEO4J_USERNAME`, `NEO4J_PASSWORD`, optional `NEO4J_DATABASE`.
(An idle Aura Free instance may be **paused** - resume it in the Aura console first.)

* every node created here is tagged `ds = "utd19-torino"`; `CLEAR_GRAPH = True` deletes **only** those
* everything uses `MERGE`: running it twice duplicates nothing
* `IncidentType {name: 'traffic_jam'}` is a **shared** node, the other City Brain datasets attach to the same one
* `Reading` nodes exist only for the stored windows (not for held-out detectors); detectors, episodes and incidents are all there

In [ ]:
_driver, NEO4J_DB = None, "neo4j"

def get_driver():
    """Create the Neo4j driver once and reuse it."""
    global _driver, NEO4J_DB
    if _driver is None:
        try:
            import neo4j  # noqa: F401
        except ImportError:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "neo4j"], check=True)
        from neo4j import GraphDatabase
        uri  = secret("NEO4J_URI", "Neo4j URI (neo4j+s://...)")
        user = secret("NEO4J_USERNAME", "Neo4j username", default="neo4j")
        pwd  = secret("NEO4J_PASSWORD", "Neo4j password", hidden=True)
        NEO4J_DB = secret("NEO4J_DATABASE", "", default="neo4j")
        drv = GraphDatabase.driver(uri, auth=(user, pwd))
        try:
            drv.verify_connectivity()
        except Exception as e:
            raise RuntimeError(f"Could not connect to Neo4j at {uri}. Is the Aura instance running (not paused)? "
                               f"Check URI / user / password. Original error: {e}")
        _driver = drv
    return _driver

def q(query, **params):
    """Run a read query, return a list of plain dicts."""
    with get_driver().session(database=NEO4J_DB) as s:
        return [r.data() for r in s.run(query, **params)]

def q_raw(query, **params):
    """Run a query and keep the driver objects (nodes / paths) - used for drawing."""
    with get_driver().session(database=NEO4J_DB) as s:
        return list(s.run(query, **params))

def rows(frame):
    """DataFrame -> list of plain dicts (NaN -> None, numpy -> python): safe to send to Neo4j."""
    return json.loads(frame.to_json(orient="records"))

def clear_graph():
    """Delete only the nodes THIS notebook created (tagged ds = DS). Shared IncidentType nodes are never touched."""
    total = 0
    with get_driver().session(database=NEO4J_DB) as s:
        while True:
            n = s.run("MATCH (n {ds: $ds}) WITH n LIMIT 5000 DETACH DELETE n RETURN count(*) AS n", ds=DS).single()["n"]
            total += n
            if n == 0:
                break
    print(f"deleted {total:,} nodes tagged ds = '{DS}'")

def load(query, data, size=1000, **extra):
    """Send `data` to Neo4j in batches; `query` must start with UNWIND $rows AS r."""
    with get_driver().session(database=NEO4J_DB) as s:
        for i in range(0, len(data), size):
            s.run(query, rows=data[i:i + size], ds=DS, **extra).consume()

def load_rel(rel, a_label, a_key, b_label, b_key, data, size=2000, props=()):
    sets = (" SET " + ", ".join(f"x.{p} = r.{p}" for p in props)) if props else ""
    load(f"UNWIND $rows AS r MATCH (a:{a_label} {{{a_key}: r.a}}), (b:{b_label} {{{b_key}: r.b}}) MERGE (a)-[x:{rel}]->(b){sets}", data, size)

def make_constraints(pairs):
    with get_driver().session(database=NEO4J_DB) as s:
        for label, key in pairs:
            s.run(f"CREATE CONSTRAINT {label.lower()}_{key} IF NOT EXISTS FOR (n:{label}) REQUIRE n.{key} IS UNIQUE").consume()

In [ ]:
# ---------------- graph tables ----------------
n_det   = rows(used.assign(speed_limit=used["limit"])[["detid", "road", "fclass", "lanes", "speed_limit", "lat", "lon"]].rename(columns={"detid": "id"}))
links   = used.dropna(subset=["linkid"]).drop_duplicates("linkid")
n_link  = rows(pd.DataFrame({"id": links.linkid.astype(int)}))
n_road  = rows(pd.DataFrame({"name": sorted(used.road.unique())}))
r_onlink = rows(used.dropna(subset=["linkid"]).assign(a=lambda x: x.detid, b=lambda x: x.linkid.astype(int))[["a", "b"]])
r_onroad = rows(links.assign(a=lambda x: x.linkid.astype(int), b=lambda x: x.road)[["a", "b"]])
r_next   = rows(neigh.rename(columns={"a": "a", "b": "b"}))
n_ep     = rows(ep[["id", "day", "start", "end", "minutes", "min_speed_ratio", "max_occ"]].rename(columns={"start": "start_at", "end": "end_at"}).round(3))
n_inc    = rows(inc[["id", "day", "minutes", "n_detectors", "n_episodes", "confirmed", "roads", "min_speed_ratio"]].round(3))
r_at_ep  = rows(ep.rename(columns={"id": "a", "detid": "b"})[["a", "b"]])
r_partof = rows(ep.rename(columns={"id": "a", "incident": "b"})[["a", "b"]])
r_isa    = [{"a": i, "b": "traffic_jam"} for i in inc.id]
n_reading = rows(train.assign(speed_ratio=train.speed_ratio_mean, occ=train.occ_mean, flow_norm=train.flow_norm_mean)
                      .rename(columns={"point_id": "id"})[["id", "state", "hour", "day", "weekend", "speed_ratio", "occ", "flow_norm"]].round(3))
r_at_rd  = rows(train.rename(columns={"point_id": "a", "detid": "b"})[["a", "b"]])
r_in_ep  = rows(train.dropna(subset=["episode"]).rename(columns={"point_id": "a", "episode": "b"})[["a", "b"]])

plan = pd.DataFrame([
    ("nodes", "Detector", len(n_det)), ("nodes", "Link", len(n_link)), ("nodes", "Road", len(n_road)),
    ("nodes", "JamEpisode", len(n_ep)), ("nodes", "Incident", len(n_inc)), ("nodes", "Reading", len(n_reading)),
    ("nodes", "IncidentType (shared)", 1),
    ("rels", "ON_LINK", len(r_onlink)), ("rels", "ON_ROAD", len(r_onroad)), ("rels", "NEXT_TO", len(r_next)),
    ("rels", "AT", len(r_at_ep) + len(r_at_rd)), ("rels", "PART_OF", len(r_partof)), ("rels", "IS_A", len(r_isa)),
    ("rels", "IN_EPISODE", len(r_in_ep)),
], columns=["kind", "name", "to_load"])
print(f"{plan[plan.kind == 'nodes'].to_load.sum():,} nodes and {plan[plan.kind == 'rels'].to_load.sum():,} relationships")
plan

In [ ]:
Q_DET  = ("UNWIND $rows AS r MERGE (d:Detector {id: r.id}) SET d.road = r.road, d.fclass = r.fclass, d.lanes = r.lanes, "
          "d.speed_limit = r.speed_limit, d.lat = r.lat, d.lon = r.lon, d.ds = $ds")
Q_LINK = "UNWIND $rows AS r MERGE (l:Link {id: r.id}) SET l.ds = $ds"
Q_ROAD = "UNWIND $rows AS r MERGE (x:Road {name: r.name}) SET x.ds = $ds"
Q_EP   = ("UNWIND $rows AS r MERGE (e:JamEpisode {id: r.id}) SET e.day = r.day, e.start_at = r.start_at, e.end_at = r.end_at, e.minutes = r.minutes, "
          "e.min_speed_ratio = r.min_speed_ratio, e.max_occ = r.max_occ, e.ds = $ds")
Q_INC  = ("UNWIND $rows AS r MERGE (i:Incident {id: r.id}) SET i.day = r.day, i.minutes = r.minutes, i.n_detectors = r.n_detectors, "
          "i.n_episodes = r.n_episodes, i.confirmed = r.confirmed, i.roads = r.roads, i.min_speed_ratio = r.min_speed_ratio, i.ds = $ds")
Q_RD   = ("UNWIND $rows AS r MERGE (x:Reading {id: r.id}) SET x.state = r.state, x.hour = r.hour, x.day = r.day, x.weekend = r.weekend, "
          "x.speed_ratio = r.speed_ratio, x.occ = r.occ, x.flow_norm = r.flow_norm, x.ds = $ds")

def load_graph():
    make_constraints([("Detector", "id"), ("Link", "id"), ("Road", "name"), ("JamEpisode", "id"), ("Incident", "id"),
                      ("Reading", "id"), ("IncidentType", "name")])
    if CLEAR_GRAPH:
        clear_graph()
    load(Q_DET, n_det); load(Q_LINK, n_link); load(Q_ROAD, n_road); load(Q_EP, n_ep); load(Q_INC, n_inc); load(Q_RD, n_reading)
    with get_driver().session(database=NEO4J_DB) as s:
        s.run("MERGE (i:IncidentType {name: 'traffic_jam'})").consume()      # shared node, no ds tag on purpose
    load_rel("ON_LINK", "Detector", "id", "Link", "id", r_onlink)
    load_rel("ON_ROAD", "Link", "id", "Road", "name", r_onroad)
    load_rel("NEXT_TO", "Detector", "id", "Detector", "id", r_next, props=("dist_m",))
    load_rel("AT", "JamEpisode", "id", "Detector", "id", r_at_ep)
    load_rel("PART_OF", "JamEpisode", "id", "Incident", "id", r_partof)
    load_rel("IS_A", "Incident", "id", "IncidentType", "name", r_isa)
    load_rel("AT", "Reading", "id", "Detector", "id", r_at_rd)
    load_rel("IN_EPISODE", "Reading", "id", "JamEpisode", "id", r_in_ep)

if RUN_NEO4J:
    get_driver()
    print("Connected to Neo4j. Database:", NEO4J_DB)
    load_graph()
    got_nodes = {r["label"]: r["n"] for r in q("MATCH (n) WHERE n.ds = $ds OR n:IncidentType RETURN labels(n)[0] AS label, count(*) AS n", ds=DS)}
    got_rels  = {r["rel"]: r["n"] for r in q("MATCH (a)-[r]->(b) WHERE a.ds = $ds RETURN type(r) AS rel, count(*) AS n", ds=DS)}
    check = plan.copy()
    check["in_neo4j"] = [got_nodes.get(n.split(" ")[0], 0) if k == "nodes" else got_rels.get(n, 0) for k, n in zip(check.kind, check.name)]
    check["ok"] = check.in_neo4j >= check.to_load
    display(check)
    assert check.ok.all(), "Some counts are lower than expected - see the table above"
    print("Graph loaded and verified.")
else:
    print("RUN_NEO4J is False - skipping the connection and the load.")

### Questions for the graph

In [ ]:
if RUN_NEO4J:
    print("Roads with the most jam episodes:")
    display(pd.DataFrame(q("""
        MATCH (e:JamEpisode)-[:AT]->(d:Detector)
        RETURN d.road AS road, count(e) AS episodes, count(DISTINCT d) AS detectors, round(avg(e.minutes)) AS avg_minutes
        ORDER BY episodes DESC LIMIT 8""")))

    print("How many jams were seen by several neighbouring detectors (real) vs one (probable glitch)?")
    display(pd.DataFrame(q("""
        MATCH (i:Incident)-[:IS_A]->(:IncidentType {name: 'traffic_jam'})
        RETURN i.confirmed AS confirmed_by_neighbours, count(i) AS incidents, round(avg(i.minutes)) AS avg_minutes,
               round(avg(i.n_detectors), 1) AS avg_detectors""")))

    print("Largest incidents:")
    display(pd.DataFrame(q("""
        MATCH (i:Incident) RETURN i.id AS incident, i.day AS day, i.minutes AS minutes, i.n_detectors AS detectors, i.roads AS roads
        ORDER BY detectors DESC, minutes DESC LIMIT 6""")))
else:
    print("RUN_NEO4J is False.")

### Explain one reading

`facts(reading_id)` collects what the graph knows around a detector-hour, `describe()` turns it into a sentence, `draw_explanation()` draws it.

In [ ]:
Q_FACTS = """
MATCH (x:Reading {id: $id})-[:AT]->(d:Detector)
OPTIONAL MATCH (x)-[:IN_EPISODE]->(e:JamEpisode)
OPTIONAL MATCH (e)-[:PART_OF]->(i:Incident)
OPTIONAL MATCH (d)-[:NEXT_TO]-(n:Detector)
RETURN x.state AS state, x.hour AS hour, x.day AS day, x.speed_ratio AS speed_ratio, x.occ AS occ, x.flow_norm AS flow_norm,
       d.id AS detector, d.road AS road, d.fclass AS fclass, d.lanes AS lanes,
       e.id AS episode, e.minutes AS ep_minutes, i.id AS incident, i.n_detectors AS n_detectors, i.minutes AS inc_minutes,
       i.confirmed AS confirmed, count(DISTINCT n) AS neighbours
LIMIT 1
"""

def facts(reading_id):
    rec = q(Q_FACTS, id=str(reading_id))
    if not rec: return None
    f = rec[0]; f["id"] = str(reading_id); return f

def describe(f):
    if f is None:
        return "(this id is not in the graph - load the graph from the same run that built Qdrant)"
    s = (f"Detector {f['detector']} on {f['road']} ({f['fclass']}, {f['lanes']} lanes, {f['neighbours']} neighbouring detectors), "
         f"{f['day']} at {f['hour']:02d}h: state {f['state']} (speed {f['speed_ratio'] * 100:.0f}% of its normal, occupancy {f['occ']:.0f}%, "
         f"road {f['flow_norm'] * 100:.0f}% full)")
    if f.get("episode"):
        s += f". Part of jam episode {f['episode']} ({f['ep_minutes']} min)"
        if f.get("incident"):
            s += (f" in incident {f['incident']}: {f['n_detectors']} detector(s) over {f['inc_minutes']} min -> "
                  + ("CONFIRMED by neighbours, a real jam" if f["confirmed"] else "seen by ONE detector only, possibly a local slowdown or sensor glitch"))
    else:
        s += ". Not part of any jam episode"
    return s + "."

Q_PATHS = """
MATCH (x:Reading {id: $id})-[:AT]->(d:Detector)
OPTIONAL MATCH p1 = (d)-[:ON_LINK]->(:Link)-[:ON_ROAD]->(:Road)
OPTIONAL MATCH p2 = (x)-[:IN_EPISODE]->(:JamEpisode)-[:PART_OF]->(:Incident)-[:IS_A]->(:IncidentType)
OPTIONAL MATCH p3 = (d)-[:NEXT_TO]-(:Detector)
RETURN x, d, p1, p2, collect(p3)[..4] AS neigh
"""
NODE_COLORS = {"Reading": "#2a78d6", "Detector": "#eda100", "Link": "#e87ba4", "Road": "#6250d6", "JamEpisode": "#eb6834",
               "Incident": "#e34948", "IncidentType": "#1baf7a"}
LAYOUT_SLOTS = {"Reading": [(-3.5, 0)], "Detector": [(-3.5, 2.0), (0.2, 0.9), (0.2, 2.5), (0.2, 4.1), (0.2, 5.7)],
                "Link": [(-3.5, 4.0)], "Road": [(-3.5, 6.0)],
                "JamEpisode": [(3.6, 0)], "Incident": [(3.6, 2.4)], "IncidentType": [(3.6, 4.8)]}

def _key(n): return getattr(n, "element_id", None) or n.id
def _text(n):
    lab, p = next(iter(n.labels)), dict(n.items())
    return {"Reading": f"{p.get('state')}\n{p.get('day', '')} {p.get('hour', '')}h", "Detector": f"detector {p.get('id')}",
            "Link": f"link {p.get('id')}", "Road": str(p.get("name", ""))[:22], "JamEpisode": f"{p.get('id')}\n{p.get('minutes')} min",
            "Incident": f"{p.get('id')}\n{p.get('n_detectors')} detectors" + ("\nCONFIRMED" if p.get("confirmed") else ""),
            "IncidentType": str(p.get("name", "")).replace("_", " ")}.get(lab, lab)

def draw_explanation(reading_id):
    recs = q_raw(Q_PATHS, id=str(reading_id))
    if not recs:
        print("reading not found"); return
    r0 = recs[0]
    G, labels, kinds = nx.Graph(), {}, {}
    def add_path(path):
        if path is None: return
        for n in path.nodes:
            k = _key(n); labels[k] = _text(n); kinds[k] = next(iter(n.labels)); G.add_node(k)
        for r in path.relationships:
            lab = r.type
            if r.type == "NEXT_TO":
                lab = f"{dict(r.items()).get('dist_m', 0):.0f} m"
            G.add_edge(_key(r.start_node), _key(r.end_node), label=lab)
    for k in ("x", "d"):
        n = r0[k]; kk = _key(n); labels[kk] = _text(n); kinds[kk] = next(iter(n.labels)); G.add_node(kk)
    G.add_edge(_key(r0["x"]), _key(r0["d"]), label="AT")
    for p in [r0["p1"], r0["p2"]] + list(r0["neigh"]):
        add_path(p)
    used_, pos = Counter(), {}
    for k in G.nodes:
        slots = LAYOUT_SLOTS[kinds[k]]
        pos[k] = slots[min(used_[kinds[k]], len(slots) - 1)]; used_[kinds[k]] += 1
    fig, ax = plt.subplots(figsize=(10, 5.8))
    nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#9a9992", width=1.8)
    nx.draw_networkx_edge_labels(G, pos, nx.get_edge_attributes(G, "label"), ax=ax, font_size=8, font_color=INK2, rotate=False,
                                 bbox=dict(boxstyle="round,pad=0.15", fc=SURF, ec="none"))
    nx.draw_networkx_nodes(G, pos, ax=ax, node_color=[NODE_COLORS[kinds[k]] for k in G], node_size=1000, edgecolors=SURF, linewidths=2)
    nx.draw_networkx_labels(G, {k: (x, y - 0.45) for k, (x, y) in pos.items()}, labels, ax=ax, font_size=8.5, font_color=INK)
    handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=k) for k, c in NODE_COLORS.items() if k in kinds.values()]
    ax.legend(handles=handles, loc="lower center", fontsize=8, ncol=7)
    ax.set_xlim(-5.4, 5.6); ax.set_ylim(-1.5, 6.9)
    ax.set_title("What the graph knows around this detector-hour"); ax.axis("off")
    plt.tight_layout(); plt.show()

if RUN_NEO4J:
    pick = train[(train.state == "jam") & train.episode.notna()]
    pick = (pick if len(pick) else train).iloc[0]
    print(describe(facts(pick.point_id)))
    draw_explanation(pick.point_id)
else:
    print("RUN_NEO4J is False.")

## 7 · Combine — Qdrant finds similar hours, Neo4j explains them

`diagnose(speed, occ, flow, base_speed, base_flow, p95_flow)` takes **one hour of 5-minute readings** (12 values each) plus that detector's normals:

1. turn the hour into the 6 numbers and encode them,
2. **Qdrant**: the 15 most similar past detector-hours and a majority vote on the state,
3. **Neo4j**: among the similar past **jam** hours, how many belonged to an incident **confirmed by neighbours**, i.e. how often this kind of reading was a real jam.

Set `RUN_COMBINE = True` (needs sections 5 and 6). The demo uses **held-out detectors**: readings the databases have never seen.
For a new city, compute `base_speed`, `base_flow` (median for that hour and weekday/weekend) and `p95_flow` from that detector's own history, then call `diagnose` with its 12 latest readings.

In [ ]:
def window_features(speed, occ, flow, base_speed, base_flow, p95_flow):
    speed, occ, flow = (np.asarray(v, float) for v in (speed, occ, flow))
    sr = np.where(base_speed > 1, speed / base_speed, 1.0)
    return {"speed_ratio_mean": sr.mean(), "speed_ratio_min": sr.min(), "occ_mean": occ.mean(), "occ_max": occ.max(),
            "flow_norm_mean": (flow / max(p95_flow, 1)).mean(), "flow_ratio_mean": np.minimum(flow / max(base_flow, 1), 5).mean()}

def vote(hits_):
    c = Counter(p["state"] for _, _, p in hits_)
    state, n = c.most_common(1)[0]
    return state, n / len(hits_), c

def diagnose(speed, occ, flow, base_speed, base_flow, p95_flow, k=K, verbose=True):
    f = window_features(speed, occ, flow, base_speed, base_flow, p95_flow)
    hits_ = qdrant_search(COLLECTION, encode(f)[0], k=k)
    state, share, counts = vote(hits_)
    out = {"features": f, "state": state, "confidence": share, "hits": hits_}
    if verbose:
        print(f"READING: speed {f['speed_ratio_mean'] * 100:.0f}% of normal (worst {f['speed_ratio_min'] * 100:.0f}%), occupancy {f['occ_mean']:.0f}% (max {f['occ_max']:.0f}%), "
              f"road {f['flow_norm_mean'] * 100:.0f}% full")
        print(f"QDRANT : {k} nearest past hours -> {dict(counts)}   verdict: {state.upper()} ({share * 100:.0f}% of neighbours)")
    if RUN_NEO4J or _driver is not None:
        fs = [facts(i) for i, _, p in hits_ if p["state"] == "jam"]
        fs = [f_ for f_ in fs if f_]
        if fs:
            conf = [f_ for f_ in fs if f_.get("incident") and f_.get("confirmed")]
            single = [f_ for f_ in fs if f_.get("incident") and not f_.get("confirmed")]
            known = len(conf) + len(single)
            out["confirmed_share"] = len(conf) / known if known else None
            if verbose and known:
                print(f"NEO4J  : of the {len(fs)} similar past JAM hours, {len(conf)} belonged to incidents confirmed by neighbouring detectors "
                      f"and {len(single)} to single-detector episodes -> "
                      + ("likely a REAL, network-wide jam" if len(conf) / known >= 0.5 else "more likely a LOCAL slowdown / sensor glitch")
                      + f" ({len(conf) / known * 100:.0f}% confirmed)")
                top = max(conf or single, key=lambda f_: (f_.get("n_detectors") or 0))
                print("         e.g.", describe(top))
        elif verbose:
            print("NEO4J  : none of the similar hours was a jam, so there is no incident to explain.")
    if verbose: print()
    return out

def raw_window(detid, day_idx, hour):
    w = d[(d.detid == detid) & (d.day_idx == day_idx) & (d.hour == hour)]
    r = w.iloc[0]
    return dict(speed=w.speed.to_numpy(), occ=w.occ.to_numpy(), flow=w.flow.to_numpy(),
                base_speed=float(r.base_speed), base_flow=float(r.base_flow), p95_flow=float(r.p95_flow))

if RUN_COMBINE:
    for st in ["jam", "slow", "free_flow", "night_empty"]:
        cand = ho[ho.state == st]
        if len(cand) == 0:
            print(f"(no held-out {st} window)\n"); continue
        r = cand.iloc[len(cand) // 2]
        print(f"=== held-out detector {int(r.detid)}, {r.day} {int(r.hour):02d}h (true state by rule: {st}) ===")
        diagnose(**raw_window(int(r.detid), int(r.day_idx), int(r.hour)))
else:
    print("RUN_COMBINE is False - skipping.")

### Final check on unseen detectors, straight through the cloud

In [ ]:
if RUN_COMBINE and len(ho):
    s = ho.sample(min(600, len(ho)), random_state=SEED).reset_index(drop=True)
    res = qdrant_search_batch(COLLECTION, encode(s), k=K)
    s["cloud"] = [vote(r)[0] for r in res]
    nn = knn_idx(encode(s), Xtr); ytr = train.state.to_numpy()
    s["offline"] = [Counter(ytr[r]).most_common(1)[0][0] for r in nn]
    print(f"{len(s):,} held-out windows")
    print(f"state consistency via Qdrant : {(s.cloud == s.state).mean() * 100:.1f}%")
    print(f"state consistency offline    : {(s.offline == s.state).mean() * 100:.1f}%")
    print(f"cloud == offline             : {(s.cloud == s.offline).mean() * 100:.1f}% of windows")
    display(pd.crosstab(s.state, s.cloud))
else:
    print("RUN_COMBINE is False - skipping.")

## What's next

* **Another city**: compute baselines for those detectors and call `diagnose`. The six features are relative to each detector's own normal, so no retraining is needed.
* **Real incidents**: if you get a few labelled accidents, mark the matching `JamEpisode` nodes, then Qdrant + Neo4j can answer *"which past episodes looked like this accident?"*
* **Join with the other datasets**: the shared `IncidentType {name:'traffic_jam'}` is where the NYC 311 calls (`traffic_signal_failure`), DOT speeds and flood data can later connect, e.g. *flood → signal failures → jams*.